# 01 · Data understanding and cleaning — DNP public investment projects

**Question behind the project:** what does Colombia's public-investment project registry (DNP, *Banco de Proyectos*) actually tell us, and what can it *not* tell us?

**Source:** [datos.gov.co · DNP-proyectos_datos_basicos](https://www.datos.gov.co/en/Econom-a-y-Finanzas/DNP-proyectos_datos_basicos/cf9k-55fw) · License CC BY-SA 4.0 · snapshot downloaded 2026-11-25.

**Output:** `data/processed/projects_clean.parquet` — original columns untouched, plus clean columns and quality flags.

> Version 1 of this project skipped this step. Several of its conclusions were artifacts of the issues documented below.

In [ ]:
import sys
sys.path.append("../src")
import pandas as pd, numpy as np
from dnp_utils import load_raw, add_clean_columns, PROCESSED, MAX_PLAUSIBLE_BENEFICIARIES, MAX_PLAUSIBLE_END_YEAR

raw = load_raw()
print(raw.shape)
raw.head(3).T

## 1. Is `Bpin` really a unique key?

In [ ]:
print("Rows:", len(raw), "| unique Bpin:", raw.Bpin.nunique())
print("Duplicated Bpin rows:", raw.Bpin.duplicated().sum())

Each row is a project (BPIN = project ID). The original notebook de-duplicated by BPIN; with this snapshot there is nothing to remove, so I **assert** uniqueness instead of silently dropping rows.

In [ ]:
assert raw.Bpin.is_unique
raw.isna().mean().round(4).sort_values(ascending=False).head(8)

## 2. Category columns: case duplicates and different schemas

In [ ]:
print(raw.TipoProyecto.value_counts().to_string())

`Pgn` and `PGN` are the same source written twice → normalised to `TypeClean`.

More importantly, **project status is not comparable across funding sources**. Look at the status × source table:

In [ ]:
d = add_clean_columns(raw)
pd.crosstab(d.EstadoProyecto, d.TypeClean)

**Finding:** `Terminado` (finished) exists **only for SGR** (regalías). For every other source the registry has no "finished" state at all. Any "% of projects finished" computed over the whole file is therefore an artifact of how the data is recorded, not a measure of performance. Version 1 reported "~5% finished" — that number is meaningless. The honest completion analysis must be restricted to SGR (notebook 02).

## 3. Money columns

In [ ]:
val = ["ValorTotalProyecto","ValorVigenteProyecto","ValorObligacionProyecto","ValorPagoProyecto"]
print("Share of projects with ValorTotal == 0:", round((d.ValorTotalProyecto==0).mean(),3))
print("\nShare with zero value, by source:")
print((d.assign(z=d.ValorTotalProyecto==0).groupby("TypeClean").z.mean().round(3)).to_string())
d[val][d.has_value].describe(percentiles=[.5,.9,.99,.999]).round(0).T

**Findings**
- About **55% of projects have `ValorTotalProyecto = 0`**. Zero here means *not valued yet* (mostly projects in formulation), not "free". They are treated as missing for any money analysis, never as zeros.
- Values are extremely **heavy-tailed** (mean ≫ median), so totals are dominated by a few mega-projects. I report medians next to sums.
- The dataset does not document currency or units. Values are assumed to be nominal Colombian pesos; projects span 2010-2030+, so they are **not inflation-adjusted** and cross-year comparisons are only indicative.
- The meaning of `Total` vs `Vigente` (current/valid value) is not documented either; I treat `Vigente` as the best-available current budget and say so wherever it is used.

## 4. Horizon (`Horizonte`) outliers

In [ ]:
print(d.HorizonYears.describe().round(1).to_string())
print("\nEnd year > %d: %d projects" % (MAX_PLAUSIBLE_END_YEAR, d.horizon_suspect.sum()))
d.loc[d.horizon_suspect, ["Bpin","Horizonte","TypeClean","EstadoProyecto"]].sort_values("Horizonte").tail(5)

Horizons ending after 2040 (up to 2108) are typing errors. They are flagged (`horizon_suspect`) and excluded from duration analysis, not deleted.

## 5. Beneficiaries (`TotalBeneficiario`)

In [ ]:
b = d.TotalBeneficiario.astype("float")
print("Colombia population ≈ 52M. Cap used:", f"{MAX_PLAUSIBLE_BENEFICIARIES:,}")
print("Zero or missing:", int(((b==0)|b.isna()).sum()))
print("Above cap      :", int((b>MAX_PLAUSIBLE_BENEFICIARIES).sum()))
print("Max value      :", f"{b.max():,.0f}")
print("Rows equal to int32 max (2,147,483,647):", int((b==2147483647).sum()))
b[b>MAX_PLAUSIBLE_BENEFICIARIES].sort_values().tail(5)

Some projects claim **more beneficiaries than the entire population** (up to ~48 *billion*). This column is user-entered and not validated. Version 1 divided money by this column to rank "efficiency" — the "most efficient" sectors were simply the ones with typos. Here: values outside `[1, 60M]` are flagged `benef_valid = False` and excluded from any per-beneficiary metric, and even valid values are treated as *indicative only*.

## 6. Save the clean table

In [ ]:
PROCESSED.mkdir(parents=True, exist_ok=True)
d.to_parquet(PROCESSED/"projects_clean.parquet", index=False)

summary = pd.Series({
 "projects": len(d),
 "with value": int(d.has_value.sum()),
 "suspect horizon": int(d.horizon_suspect.sum()),
 "invalid beneficiaries": int((~d.benef_valid.fillna(False)).sum()),
 "finished (all SGR)": int((d.StateGroup=="Finished").sum()),
})
summary

### Cleaning decisions (audit trail)

| # | Issue | Decision |
|---|---|---|
| 1 | `Pgn`/`PGN` case duplicates | Normalised into `TypeClean` |
| 2 | Status schema differs by source | Created `StateGroup`; completion analysed on SGR only |
| 3 | 55% zero `ValorTotal` | Treated as "not valued", excluded from money stats |
| 4 | Horizons up to year 2108 | Flagged `horizon_suspect`, excluded from duration stats |
| 5 | Impossible beneficiaries | Flagged `benef_valid`, excluded from per-beneficiary metrics |
| 6 | No units/currency/inflation info | Documented as caveat; nominal values only |